# Automated PDE Generator: Architecture Report

This notebook documents the full pipeline that derives layer-averaged
shallow water equations from the 3D Incompressible Navier-Stokes (INS)
equations using Galerkin projection with arbitrary vertical basis functions.

**Pipeline overview:**

| Phase | What it does | Key class |
|-------|-------------|-----------|
| 1 | Abstract INS + Galerkin projection + BCs | `INSBase`, `GalerkinProjection`, `BoundaryConditions` |
| 2 | Inject piecewise basis with Heaviside windowing | `LayeredAnsatz` |
| 3 | Custom integration engine (avoids SymPy hangs) | `PiecewiseIntegrator` |
| 4 | Route interface deltas: boundary vs internal | `InterfaceRouter` |
| 5 | Assemble into Model-compatible class | `GeneratedShallowModel` |

In [ ]:
import sympy as sp
from sympy import (
    Symbol, Function, Derivative, Integral, Rational,
    Heaviside, DiracDelta, S, Matrix, sqrt,
)
sp.init_printing()

: 

---
## Phase 1: Starting from Physics

### 1.1 The 3D Incompressible Navier-Stokes Equations

We start from the full 3D INS in continuous form. All fields are
generic SymPy `Function` objects — no discretization yet.

In [ ]:
from zoomy_core.model.models.pde_generator import INSBase

ins = INSBase(dimension=1)
print(f"Fields: u={ins.u}, w={ins.w}, p={ins.p}")
print(f"Parameters: rho={ins.rho}, nu={ins.nu}, g={ins.g}")

**Continuity equation:** $\nabla \cdot \mathbf{u} = 0$

In [ ]:
ins.continuity()

**X-momentum equation:** $\partial_t u + \partial_x(u^2) + \partial_z(uw) + \frac{1}{\rho}\partial_x p = 0$

In [ ]:
ins.x_momentum()

**Hydrostatic pressure relation:** $\partial_z p + \rho g = 0$

In [ ]:
ins.hydrostatic_pressure()

The 2D case adds $v(t,x,y,z)$ and a $y$-momentum equation:

In [ ]:
ins2d = INSBase(dimension=2)
print(f"2D fields: u={ins2d.u}, v={ins2d.v}, w={ins2d.w}")
display(ins2d.continuity())
display(ins2d.y_momentum())

---
### 1.2 Abstract Galerkin Projection

We project the INS onto vertical test functions $\phi_i(\zeta)$ with
integration weight $c(\zeta)$, where $\zeta = (z-b)/H \in [0,1]$:

$$\int_0^1 \text{PDE} \cdot c(\zeta) \cdot \phi_i(\zeta) \cdot H \, d\zeta = 0$$

The weight $c(\zeta)$ allows non-standard inner products (e.g. $c = \sqrt{1-\zeta^2}$
for Chebyshev). For Legendre, $c=1$.

At this stage everything stays **abstract** — no specific basis, no layers.
Integrals are kept as unevaluated `Integral` objects.

In [ ]:
from zoomy_core.model.models.pde_generator import GalerkinProjection

proj = GalerkinProjection(ins)
print(f"Bathymetry: b = {proj.b}")
print(f"Total depth: H = {proj.H}")
print(f"Reference coord: zeta = {proj.zeta}")
print(f"Test function: phi_i = {proj.phi_i}")
print(f"Weight: c = {proj.c_weight}")
print(f"Coordinate map: z = {proj.z_of_zeta()}")

**Projected continuity equation:**

Integration by parts on the $\partial_z w$ term produces boundary terms
at $\zeta=0$ (bottom) and $\zeta=1$ (surface), plus a volume integral.

In [ ]:
eq_cont = proj.project_continuity()
print(f"Equation: {eq_cont.name}")
print("\nVolume integral terms:")
display(eq_cont.volume)
print("\nBoundary at zeta=1 (surface):")
display(eq_cont.boundary_top)
print("\nBoundary at zeta=0 (bottom):")
display(eq_cont.boundary_bottom)

**Projected x-momentum:**

Same structure — the $\partial_z(uw)$ term produces boundary contributions
where the kinematic BCs and surface/bottom stresses will enter.

In [ ]:
eq_mom = proj.project_momentum(component_index=0)
print(f"Equation: {eq_mom.name}")
print("\nVolume terms:")
display(eq_mom.volume)
print("\nBoundary (surface):")
display(eq_mom.boundary_top)
print("\nBoundary (bottom):")
display(eq_mom.boundary_bottom)

---
### 1.3 Physical Boundary Conditions

Before any discretization, we apply the physics at the top and bottom
of the water column.

**Kinematic BCs** (zero relative mass flux):
- Bottom ($z=b$): $w_b = \partial_t b + u_b \partial_x b$
- Surface ($z=b+H$): $w_s = \partial_t(b+H) + u_s \partial_x(b+H)$

**Dynamic BCs:**
- Surface stress $\tau_s$ (wind)
- Bottom stress $\tau_b$ (friction)
- Atmospheric pressure $p_{atm}$

**Hydrostatic pressure:**
$p = p_{atm} + \rho g (b+H-z)$, so $\frac{1}{\rho}\partial_x p = \frac{1}{\rho}\partial_x p_{atm} + g\,\partial_x(b+H)$

In [ ]:
from zoomy_core.model.models.pde_generator import BoundaryConditions

bc = BoundaryConditions(proj)

print("Kinematic BC at bottom:")
display(bc.kinematic_bottom())

print("\nKinematic BC at surface:")
display(bc.kinematic_surface())

print("\nHydrostatic pressure gradient (x-direction):")
display(bc.hydrostatic_pressure_gradient(ins.x))

print(f"\nStress symbols: tau_bx={bc.tau_bx}, tau_sx={bc.tau_sx}")

Applying BCs replaces the abstract $w_s$, $w_b$ in the boundary terms
and injects the stress and pressure contributions:

In [ ]:
raw_eqs = proj.project_all()
final_eqs = bc.apply_all(raw_eqs)

for eq in final_eqs:
    print(f"\n--- {eq.name} ---")
    print("Boundary (surface) after BC application:")
    display(eq.boundary_top)
    if eq.pressure_gradient is not None:
        print("Hydrostatic pressure gradient:")
        display(eq.pressure_gradient)

**User perspective on material models:**

The stress symbols $\tau_{bx}$, $\tau_{sx}$ are placeholders. A user defines
their friction law by substituting concrete expressions:

| Friction model | Bottom stress $\tau_b$ |
|---------------|----------------------|
| No-slip | $\tau_b = \rho \nu \, u_b / h$ |
| Chezy | $\tau_b = \rho \, u_b |u_b| / C^2$ |
| Manning | $\tau_b = \rho g n^2 u_b |u_b| / h^{1/3}$ |
| Navier slip | $\tau_b = \rho \, u_b / \lambda$ |

These are injected as source terms in the `GeneratedShallowModel` via
methods like `newtonian()`, `chezy()`, `slip()` — following the same
pattern as `ShallowMomentsTopo`.

---
## Phase 2: Basis Injection with Heaviside Windowing

Now we map the abstract equations into a concrete discretization.
The vertical velocity is expanded as:

$$u(t,x,\zeta) = \sum_{k=0}^{N-1} \sum_{j=0}^{L} u_{k,j}(t,x) \, \phi_j(\zeta_{\text{local}}) \, W_k(\zeta)$$

where:
- $k$ = layer index, $j$ = basis function index
- $\zeta_{\text{local}} = (\zeta - \zeta_k) / \Delta\zeta_k$ maps into each layer
- $W_k(\zeta) = H(\zeta - \zeta_k) - H(\zeta - \zeta_{k+1})$ is the Heaviside window

In [ ]:
from zoomy_core.model.models.pde_generator import LayeredAnsatz
from zoomy_core.model.models.basisfunctions import Legendre_shifted, Monomials

# Single layer, linear basis (P1)
basis = Legendre_shifted(level=1)
la = LayeredAnsatz(n_layers=1, basis=basis, dimension=1)

print("Layer interfaces:", la.layer_interfaces)
print("DOFs:", la.get_all_dof_symbols("u"))
print("\nFull ansatz (1 layer, no Heaviside needed):")
display(la.full_ansatz("u"))

With **two layers**, Heaviside windows appear:

In [ ]:
la2 = LayeredAnsatz(n_layers=2, basis=Monomials(level=0), dimension=1)
zeta = la2.zeta

print("Layer interfaces:", la2.layer_interfaces)
print("\nWindow functions:")
for k in range(2):
    print(f"  W_{k} =", la2.window_function(k))

print("\nFull piecewise ansatz:")
display(la2.full_ansatz("u"))

### Derivatives produce DiracDelta at interfaces

When we differentiate the Heaviside-windowed ansatz, the derivative of
$H(\zeta - \zeta_k)$ gives $\delta(\zeta - \zeta_k)$. These represent
**jumps** in the solution at layer interfaces.

In [ ]:
deriv = la2.derivative_ansatz("u")
print("d/dzeta [u_ansatz] =")
display(sp.expand(deriv))

The four delta terms are:
- $u_{0,0}\,\delta(\zeta)$ — bottom boundary
- $-u_{0,0}\,\delta(\zeta - 1/2) + u_{1,0}\,\delta(\zeta - 1/2)$ — **internal interface jump**
- $-u_{1,0}\,\delta(\zeta - 1)$ — top boundary

The internal jump is $(u_{1,0} - u_{0,0})\,\delta(\zeta - 1/2)$: the velocity
discontinuity between layers.

---
## Phase 3: Custom Integration Engine

SymPy's native `integrate` with `Piecewise`/`Heaviside` causes AST explosion
and hangs for multi-layer expressions. The `PiecewiseIntegrator` avoids this by:

1. **Splitting** the expression into smooth (Heaviside) and singular (DiracDelta) parts
2. **Static collapse**: per layer chunk, evaluate Heaviside at the midpoint to get 0 or 1,
   then integrate the resulting smooth polynomial
3. **Delta sifting**: apply $\int f(z)\,\delta(z-z_k)\,dz = f(z_k)$

In [ ]:
from zoomy_core.model.models.pde_generator import PiecewiseIntegrator

z = Symbol("z")

# Two-layer domain [0, 1/2, 1]
integrator = PiecewiseIntegrator(z, [S.Zero, Rational(1, 2), S.One])

**Example 1:** Integrate a Heaviside-windowed function (smooth part).

In [ ]:
W0 = Heaviside(z) - Heaviside(z - Rational(1, 2))
expr_smooth = z**2 * W0  # z^2 on [0, 1/2], 0 on [1/2, 1]
result = integrator.integrate(expr_smooth)
print(f"integral of z^2 * W0 = {result}  (expected: 1/24)")

**Example 2:** DiracDelta sifting at an interface.

In [ ]:
f = 3*z + 1
expr_delta = f * DiracDelta(z - Rational(1, 2))
result = integrator.integrate(expr_delta)
print(f"integral of (3z+1) * delta(z-1/2) = {result}  (expected: 5/2)")

**Example 3:** Mixed smooth + delta (the real use case).

In [ ]:
expr_mixed = z * W0 + 5 * DiracDelta(z - Rational(1, 2))
result = integrator.integrate(expr_mixed)
print(f"Mixed integral = {result}  (expected: 1/8 + 5 = 41/8)")

### Integration of the piecewise ansatz

The integrator handles the full ansatz correctly.
For a 2-layer P0 ansatz, $\int_0^1 u\,d\zeta$ should give the
depth-weighted average $(u_{0,0} + u_{1,0})/2$:

In [ ]:
u_ansatz = la2.full_ansatz("u")
integrator_full = PiecewiseIntegrator(la2.zeta, la2.layer_interfaces)
result = integrator_full.integrate(u_ansatz)
display(result)
print("(This is the depth-averaged velocity)")

The "mass matrix" $\int_0^1 u^2\,d\zeta$ gives per-layer contributions:

In [ ]:
result_mass = integrator_full.integrate(u_ansatz * u_ansatz)
display(sp.expand(result_mass))

---
## Phase 4: Interface Routing

The `InterfaceRouter` classifies each DiracDelta term:

| Location | Classification | Action |
|----------|---------------|--------|
| $\zeta = 0$ (bottom) | Physical boundary | Replace with bottom BC ($\tau_b$, no-slip, etc.) |
| $\zeta = 1$ (surface) | Physical boundary | Replace with surface BC ($\tau_s$, wind stress, etc.) |
| $0 < \zeta_k < 1$ | Internal interface | Tag as **numerical flux** for Riemann solver |

In [ ]:
from zoomy_core.model.models.pde_generator import InterfaceRouter

# 3-layer setup: interfaces at 0, 1/3, 2/3, 1
interfaces_3 = [Rational(k, 3) for k in range(4)]
router = InterfaceRouter(interfaces_3, z)

# An expression with deltas at all interfaces
expr_deltas = (
    2 * DiracDelta(z)                    # boundary (bottom)
    + 7 * DiracDelta(z - Rational(1, 3)) # internal
    + 3 * DiracDelta(z - Rational(2, 3)) # internal
    + 5 * DiracDelta(z - 1)              # boundary (top)
    + z**2                                # smooth volume term
)

classified = router.classify_delta_terms(expr_deltas)

print(f"Smooth part: {classified.smooth}")
print(f"\nBoundary deltas ({classified.n_boundary}):")
for j in classified.boundary:
    print(f"  {j}")
print(f"\nInternal deltas ({classified.n_internal}):")
for j in classified.internal:
    print(f"  {j}")

**Applying physical BCs** replaces boundary deltas with user-defined conditions:

In [ ]:
tau_b = Symbol("tau_b")  # bottom friction
tau_s = Symbol("tau_s")  # surface wind stress

result_with_bcs = router.apply_physical_bcs(
    classified,
    bc_bottom=tau_b,
    bc_top=tau_s,
)
print("After applying BCs:")
display(result_with_bcs)
print("\nThe internal deltas at zeta=1/3 and zeta=2/3 are NOT touched.")
print("They become Riemann solver inputs (Phase 5).")

---
## Phase 5: The Generated Model

The `GeneratedShallowModel` assembles everything into a `Model`-compatible
class with `flux()`, `hydrostatic_pressure()`, `nonconservative_matrix()`,
`source()`, and `eigenvalues()`.

### Classical SWE: `n_layers=1, level=0`

In [ ]:
from zoomy_core.model.models.generated_shallow_model import GeneratedShallowModel

m_swe = GeneratedShallowModel(n_layers=1, level=0, dimension=1)
b, h, mu, mv, hinv = m_swe.get_primitives()

print("State vector Q =", list(m_swe.variables.values()))
print("Primitive: u =", mu[0][0], " (= hu / h)")

In [ ]:
print("Flux F(Q):")
display(m_swe.flux().tomatrix())

print("\nHydrostatic pressure F_p(Q):")
display(m_swe.hydrostatic_pressure().tomatrix())

This is exactly the classical SWE: $F = [hu,\; hu^2/h]$, $F_p = [0,\; g h^2/2]$.

In [ ]:
print("Eigenvalues:")
for ev in m_swe.eigenvalues():
    display(ev)

Eigenvalues: $0$ (passive bathymetry), $u \pm \sqrt{g\,e_z\,h}$ (gravity waves).

### Shallow Moments: `n_layers=1, level=1`

Adding a linear velocity profile $u(\zeta) = \alpha_0 P_0(\zeta) + \alpha_1 P_1(\zeta)$.

In [ ]:
m_sm = GeneratedShallowModel(n_layers=1, level=1, dimension=1)
b, h, mu, mv, hinv = m_sm.get_primitives()

print("State Q =", list(m_sm.variables.values()))
print("Primitives: alpha =", mu[0])

In [ ]:
print("Flux:")
display(m_sm.flux().tomatrix())

In [ ]:
nc = m_sm.nonconservative_matrix()
nc_x = Matrix([[nc[r, c, 0] for c in range(m_sm.n_variables)]
               for r in range(m_sm.n_variables)])
print("Non-conservative matrix B_x:")
display(nc_x)

### Algebraic Verification

The generated flux **exactly matches** the hand-derived `ShallowMomentsTopo`:

In [ ]:
from zoomy_core.model.models.shallow_moments_topo import ShallowMomentsTopo

gen = GeneratedShallowModel(n_layers=1, level=1, dimension=1)
ref = ShallowMomentsTopo(level=1, dimension=1)

subs = {}
for i in range(gen.n_variables):
    gv, rv = gen.variables[i], ref.variables[i]
    if gv != rv:
        subs[gv] = rv
for k in gen.parameters.keys():
    gp = gen.parameters[k]
    if ref.parameters.contains(k):
        rp = ref.parameters[k]
        if gp != rp:
            subs[gp] = rp

F_gen = gen.flux()
F_ref = ref.flux()

print("Flux difference (generated - reference):")
all_zero = True
for i in range(gen.n_variables):
    diff = sp.simplify(sp.expand(F_gen[i, 0].subs(subs) - F_ref[i, 0]))
    status = "OK" if diff == 0 else f"MISMATCH: {diff}"
    print(f"  Row {i}: {status}")
    if diff != 0:
        all_zero = False

if all_zero:
    print("\nAll flux components match exactly.")

### Multi-layer: `n_layers=2, level=0`

Two layers with constant velocity per layer.

In [ ]:
m_ml = GeneratedShallowModel(n_layers=2, level=0, dimension=1)
b, h, mu, mv, hinv = m_ml.get_primitives()

print("State Q =", list(m_ml.variables.values()))
print(f"Layer 0 velocity: {mu[0][0]}")
print(f"Layer 1 velocity: {mu[1][0]}")

print("\nFlux:")
display(m_ml.flux().tomatrix())

The mass flux row is $F_h = h\left(\frac{u_{0,0}}{2} + \frac{u_{1,0}}{2}\right)$
— the depth-weighted average of layer velocities.

Each layer's momentum flux is $h \cdot u_k^2 / 2$ (weighted by layer fraction $1/N$).

---
## What is Still Missing

### 1. Interface fluxes for multi-layer (Phase 5 stub architecture)

For `n_layers > 1`, the derivative of the Heaviside-windowed ansatz produces
DiracDelta terms at internal interfaces. These represent the **vertical exchange**
between layers (the $G_{k+1/2}$ terms in multi-layer formulations).

The `InterfaceRouter` correctly identifies and classifies them, but they are
**not yet wired** into the `GeneratedShallowModel`. The plan is:

- Add `interface_flux()` method that returns symbolic expressions with
  **stub function calls** like `riemann_solve(Q_left, Q_right)`
- The stubs have a clear interface (inputs/outputs) that each backend
  (NumPy, JAX, C) implements concretely
- This keeps the symbolic model backend-agnostic

### 2. Numerical eigenvalues for multi-layer

For `n_layers >= 2`, eigenvalues of the quasilinear matrix involve quartic+
polynomials with ugly algebraic roots. In practice:
- Use **numerical eigenvalue computation** at runtime (per cell, per timestep)
- For Rusanov flux, only need `max|lambda|` — no closed form needed

### 3. Viscous interface terms

The vertical diffusion $-\nu \partial_{zz} u$ also produces interface terms
when differentiated through the Heaviside windows. These should become
**viscous flux stubs** at interfaces.

### 4. Higher-order basis verification

Level 0 and 1 are verified against `ShallowMomentsTopo`. Level 2+ should
also match but tests are slow (large eigenvalue computations).

### 5. Chebyshev and other weighted bases

The framework supports the weight function $c(\zeta)$ in the projection, and
the `Chebyshevu` basis from `basisfunctions.py`. Not yet tested end-to-end
through the generator.

---
## Summary: Architecture Diagram

```
INSBase (3D continuous PDEs)
    |
    v
GalerkinProjection (abstract integrals with phi_i, c(zeta))
    |
    v
BoundaryConditions (kinematic BCs, stresses, hydrostatic pressure)
    |                           |
    v                           v
LayeredAnsatz              InterfaceRouter
(Heaviside windows,        (boundary vs internal
 basis polynomials,         delta classification,
 per-layer DOFs)            physical BC overrides)
    |                           |
    v                           v
PiecewiseIntegrator        [Phase 5: stub Riemann solver]
(static Heaviside collapse,     (TODO)
 DiracDelta sifting)            |
    |                           |
    +----------+----------------+
               |
               v
    GeneratedShallowModel (Model-compatible)
       flux(), source(), eigenvalues(), ...
       interface_flux()  [TODO]
```